## 2026 EY AI & Data Challenge — Landsat Data Extraction (MULTI-BUFFER)

This notebook extracts Landsat surface reflectance bands at **four spatial buffer sizes** around each sampling location:

| Buffer | Radius | Purpose |
|---|---|---|
| 50 m | ~1 pixel | Tight in-stream signal, minimal land mixing |
| 100 m | Default | Baseline used in Opt 15 (score 0.346) |
| 150 m | ~2 pixels | Captures near-bank vegetation/runoff context |
| 200 m | ~3 pixels | Wider catchment representation |

**Bands extracted per buffer:** `nir`, `green`, `swir16`, `swir22`  
**Derived indices:** `NDMI`, `MNDWI`

**Strategy:** 4 batches × 4 buffers = 16 cells for training; 1 batch × 4 buffers = 4 cells for validation.  
Each cell runs ≤50 locations — safe against API timeouts (~10–15 min per cell).

**Output files:**
- `landsat_features_training_50m.csv`
- `landsat_features_training_150m.csv`
- `landsat_features_training_200m.csv`
- `landsat_features_validation_50m.csv`
- `landsat_features_validation_150m.csv`
- `landsat_features_validation_200m.csv`


### Load In Dependencies
After the first run, **restart the kernel** before continuing.

### ⚠️ Action Required: Add Packages via Snowflake UI

Snowflake Notebooks do not support `!pip install`. Please add the following packages using the **Packages** dropdown at the top of this page:

**Required Packages:**
- `pystac`
- `pystac-client`
- `planetary-computer`
- `xarray`
- `zarr`
- `adlfs`
- `dask`
- `numcodecs`
- `scipy`
- `tqdm`
- `pandas`
- `numpy`
- `odc-stac` (for Landsat notebook)
- `rioxarray` (for Landsat notebook)

> [!IMPORTANT]
> After adding packages, you may need to **Restart Kernel** from the menu at the top.

In [5]:
import requests
resp = requests.get("https://planetarycomputer.microsoft.com/api/stac/v1")
print(resp.status_code)


200


In [6]:
import snowflake
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.session import Session
import requests
import warnings
import numpy as np
import pandas as pd
import rioxarray
from tqdm import tqdm
import os

try:
    session = get_active_session()
    print('✅ Linked to active Snowflake session')
except:
    try:
        session = Session.builder.getOrCreate()
        print('✅ Session created via builder')
    except:
        print('❌ No active session found. Ensure a Warehouse is selected.')

tqdm.pandas()
warnings.filterwarnings('ignore')
print('✅ Imports complete (using requests + rioxarray strategy)')

❌ No active session found. Ensure a Warehouse is selected.
✅ Imports complete (using requests + rioxarray strategy)


### Helper Functions

The key change from the original notebook: `compute_Landsat_values` now accepts a **`buffer_m`** parameter (buffer size in metres). The degree-equivalent buffer is computed from `buffer_m / 110_000.0`.

In [ ]:
# ── Helper Functions ──────────────────────────────────────────────────────────

def sign_url(href):
    """Sign a Planetary Computer URL via the SAS API."""
    url = f"https://planetarycomputer.microsoft.com/api/sas/v1/sign?href={href}"
    return requests.get(url).json()["href"]

def compute_Landsat_values(row, buffer_m=100.0):
    """Search for Landsat items via requests and compute median values using rioxarray."""
    lat  = row['Latitude']
    lon  = row['Longitude']
    date = pd.to_datetime(row['Sample Date'], dayfirst=True, errors='coerce')

    # Convert metres → decimal degrees (equatorial approximation)
    bbox_size = buffer_m / 110_000.0
    bbox = [
        lon - bbox_size / 2,
        lat - bbox_size / 2,
        lon + bbox_size / 2,
        lat + bbox_size / 2
    ]

    # ── 1. Search (requests) ──
    stac_api = "https://planetarycomputer.microsoft.com/api/stac/v1/search"
    search_params = {
        "collections": ["landsat-c2-l2"],
        "bbox": bbox,
        "datetime": "2011-01-01/2015-12-31",
        "query": {"eo:cloud_cover": {"lt": 20}},
        "limit": 50
    }
    
    try:
        resp = requests.post(stac_api, json=search_params).json()
        items = resp.get('features', [])
        
        if not items:
            return pd.Series({"nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan})
        
        # Pick closest item to sample date
        sample_dt = date if date.tzinfo else date.tz_localize("UTC")
        items = sorted(
            items, 
            key=lambda x: abs(pd.to_datetime(x['properties']['datetime']).tz_convert("UTC") - sample_dt)
        )
        item = items[0]
        
        # ── 2. Load Bands (rioxarray) ──
        bands = {"nir": "nir08", "green": "green", "swir16": "swir16", "swir22": "swir22"}
        results = {}
        
        for b_name, b_key in bands.items():
            href = item['assets'][b_key]['href']
            signed_href = sign_url(href)
            
            # Load and clip to bbox
            da = rioxarray.open_rasterio(signed_href).rio.clip_box(*bbox)
            val = da.median(skipna=True).values.item()
            results[b_name] = float(val) if val != 0 else np.nan
            
        return pd.Series(results)
        
    except Exception as e:
        return pd.Series({"nir": np.nan, "green": np.nan, "swir16": np.nan, "swir22": np.nan})

def add_indices(df):
    """Compute NDMI and MNDWI."""
    df = df.copy()
    eps = 1e-10
    df['NDMI']  = (df['nir'] - df['swir16']) / (df['nir'] + df['swir16'] + eps)
    df['MNDWI'] = (df['green'] - df['swir16']) / (df['green'] + df['swir16'] + eps)
    return df

def extract_and_save(source_df, buffer_m, batch_start, batch_end, tag='training'):
    batch = source_df.iloc[batch_start:batch_end].reset_index(drop=True)
    print(f"🚀 Extracting {buffer_m}m buffer | {tag} | rows {batch_start}–{batch_end}")

    features = batch.progress_apply(lambda row: compute_Landsat_values(row, buffer_m=buffer_m), axis=1)
    features = add_indices(features)
    
    # Add merge keys
    features['Latitude']  = batch['Latitude']
    features['Longitude'] = batch['Longitude']
    features['Sample Date'] = batch['Sample Date']
    
    fname = f"landsat_{buffer_m}m_{tag}_batch_{batch_start:04d}_{batch_end:04d}.csv"
    features.to_csv(fname, index=False)
    print(f"  ✅ Saved {fname}")
    return features

print("✅ Helper functions loaded (Native requests + rioxarray fallback)")

### Load Base Data

In [ ]:
Water_Quality_df = pd.read_csv('water_quality_training_dataset.csv')
Validation_df    = pd.read_csv('submission_template.csv')

# Convert to datetime for proper uniqueness check
Water_Quality_df['Sample Date'] = pd.to_datetime(Water_Quality_df['Sample Date'], dayfirst=True, errors='coerce')
Validation_df['Sample Date']    = pd.to_datetime(Validation_df['Sample Date'], dayfirst=True, errors='coerce')

# To save API time, we group by Location AND Year-Month
# Landsat resolution is ~16 days, so one sample per month is usually enough
# and we can reuse that value for all samples at that location in that month.
Water_Quality_df['YearMonth'] = Water_Quality_df['Sample Date'].dt.to_period('M')
Validation_df['YearMonth']    = Validation_df['Sample Date'].dt.to_period('M')

train_locs_df = Water_Quality_df[['Latitude','Longitude','Sample Date','YearMonth']].drop_duplicates(subset=['Latitude','Longitude','YearMonth']).reset_index(drop=True)
val_locs_df   = Validation_df[['Latitude','Longitude','Sample Date','YearMonth']].drop_duplicates(subset=['Latitude','Longitude','YearMonth']).reset_index(drop=True)

print(f"Total training rows: {len(Water_Quality_df)}")
print(f"Unique Location-Month pairs (Training):   {len(train_locs_df)}")
print(f"Unique Location-Month pairs (Validation): {len(val_locs_df)}")

print('\n🚀 Note: We are extracting one Landsat scene per Location per Month to cover the full temporal variation while staying efficient.')

---
## ① Automated Training Extraction — All Buffers (50m, 150m, 200m)

Since we now have **6404 unique location-month pairs**, manual batching is too slow.  
The cell below runs a loop that processes 50 locations at a time and saves progress to CSV files.  

**How to use:**
1. Run the cell below.
2. It will automatically check for existing files and skip they are already done (Checkpointing).
3. You can stop and restart the cell at any time.

In [ ]:
# ── 50m BUFFER EXTRACTION (AUTO-RESUME & PERSISTENT) ────────────────────────

def run_extraction_50m(df, batch_size=250):
    buffer_m = 50
    table_name = f"LANDSAT_STAGING_TRAINING_{buffer_m}M"
    total_rows = len(df)
    
    print(f"🏁 Starting 50m extraction for {total_rows} rows.")
    
    # 📊 CHECKPOINT: Check Snowflake table to see where we left off
    already_processed = 0
    try:
        count_df = session.sql(f"SELECT COUNT(*) FROM {table_name}").to_pandas()
        already_processed = int(count_df.iloc[0, 0])
        print(f"  ✅ Resume Point: Found {already_processed} rows already in {table_name}.")
    except Exception:
        print(f"  🆕 Table {table_name} not found. Starting from row 0.")

    for start in range(0, total_rows, batch_size):
        # Skip what's already in the cloud table
        if start < already_processed:
            continue
            
        end = min(start + batch_size, total_rows)
        
        try:
            # 1. Extract
            batch_results = extract_and_save(df, buffer_m, start, end, tag='training')
            
            # 2. 🔒 Perist to Snowflake immediately (Cloud Save)
            session.write_pandas(batch_results, table_name, auto_create_table=True, overwrite=False)
            print(f"  💾 Batch {start}-{end} synced to Snowflake.")
            
        except Exception as e:
            print(f"  ❌ Error at batch {start}: {str(e)}")
            continue

    # 3. Final Save to CSV (for downloading)
    print(f"\n🎉 50m Extraction Complete. Finalizing CSV...")
    final_df = session.table(table_name).to_pandas()
    final_df.to_csv(f"landsat_features_training_{buffer_m}m.csv", index=False)
    print(f"  📂 File Ready: landsat_features_training_{buffer_m}m.csv ({len(final_df)} rows)")

run_extraction_50m(train_locs_df)

# ── 150m BUFFER EXTRACTION (AUTO-RESUME & PERSISTENT) ──────────────────────


In [ ]:
# ── 150m BUFFER EXTRACTION (AUTO-RESUME & PERSISTENT) ──────────────────────

def run_extraction_150m(df, batch_size=250):
    buffer_m = 150
    table_name = f"LANDSAT_STAGING_TRAINING_{buffer_m}M"
    total_rows = len(df)
    
    print(f"🏁 Starting 150m extraction for {total_rows} rows.")
    
    # 📊 CHECKPOINT: Check Snowflake table to see where we left off
    already_processed = 0
    try:
        count_df = session.sql(f"SELECT COUNT(*) FROM {table_name}").to_pandas()
        already_processed = int(count_df.iloc[0, 0])
        print(f"  ✅ Resume Point: Found {already_processed} rows already in {table_name}.")
    except Exception:
        print(f"  🆕 Table {table_name} not found. Starting from row 0.")

    for start in range(0, total_rows, batch_size):
        # Skip what's already in the cloud table
        if start < already_processed:
            continue
            
        end = min(start + batch_size, total_rows)
        
        try:
            # 1. Extract
            batch_results = extract_and_save(df, buffer_m, start, end, tag='training')
            
            # 2. 🔒 Sync to Snowflake immediately (Cloud Save)
            session.write_pandas(batch_results, table_name, auto_create_table=True, overwrite=False)
            print(f"  💾 Batch {start}-{end} synced to Snowflake.")
            
        except Exception as e:
            print(f"  ❌ Error at batch {start}: {str(e)}")
            continue

    # 3. Final Save to CSV (for downloading)
    print(f"\n🎉 150m Extraction Complete. Finalizing CSV...")
    final_df = session.table(table_name).to_pandas()
    final_df.to_csv(f"landsat_features_training_{buffer_m}m.csv", index=False)
    print(f"  📂 File Ready: landsat_features_training_{buffer_m}m.csv ({len(final_df)} rows)")

run_extraction_150m(train_locs_df)

In [ ]:
# ── 200m BUFFER EXTRACTION (AUTO-RESUME & PERSISTENT) ──────────────────────

def run_extraction_200m(df, batch_size=250):
    buffer_m = 200
    table_name = f"LANDSAT_STAGING_TRAINING_{buffer_m}M"
    total_rows = len(df)
    
    print(f"🏁 Starting 200m extraction for {total_rows} rows.")
    
    # 📊 CHECKPOINT: Check Snowflake table to see where we left off
    already_processed = 0
    try:
        count_df = session.sql(f"SELECT COUNT(*) FROM {table_name}").to_pandas()
        already_processed = int(count_df.iloc[0, 0])
        print(f"  ✅ Resume Point: Found {already_processed} rows already in {table_name}.")
    except Exception:
        print(f"  🆕 Table {table_name} not found. Starting from row 0.")

    for start in range(0, total_rows, batch_size):
        # Skip what's already in the cloud table
        if start < already_processed:
            continue
            
        end = min(start + batch_size, total_rows)
        
        try:
            # 1. Extract
            batch_results = extract_and_save(df, buffer_m, start, end, tag='training')
            
            # 2. 🔒 Sync to Snowflake immediately (Cloud Save)
            session.write_pandas(batch_results, table_name, auto_create_table=True, overwrite=False)
            print(f"  💾 Batch {start}-{end} synced to Snowflake.")
            
        except Exception as e:
            print(f"  ❌ Error at batch {start}: {str(e)}")
            continue

    # 3. Final Save to CSV (for downloading)
    print(f"\n🎉 200m Extraction Complete. Finalizing CSV...")
    final_df = session.table(table_name).to_pandas()
    final_df.to_csv(f"landsat_features_training_{buffer_m}m.csv", index=False)
    print(f"  📂 File Ready: landsat_features_training_{buffer_m}m.csv ({len(final_df)} rows)")

run_extraction_200m(train_locs_df)

In [ ]:
from IPython.display import HTML, display

# List of files you want to generate download links for
csv_files = [
    "landsat2_features_training_50m.csv",
    "landsat_features_training_150m.csv",
    "landsat_features_training_200m.csv"
]

print("🚀 Click the links below to download your CSV files:")
for file in csv_files:
    # On Snowflake, we assume the file exists if the extraction finished.
    # We use a direct HTML link which is compatible with most notebook environments.
    html = f'<a href="{file}" target="_blank">Download {file}</a>'
    display(HTML(html))

---
## ④ Validation — All Buffers
200 validation locations = 1 batch per buffer size.

In [ ]:
extract_and_save(val_locs_df, buffer_m=50,  batch_start=0, batch_end=len(val_locs_df), tag='validation')

In [ ]:
extract_and_save(val_locs_df, buffer_m=150, batch_start=0, batch_end=len(val_locs_df), tag='validation')

In [ ]:
extract_and_save(val_locs_df, buffer_m=200, batch_start=0, batch_end=len(val_locs_df), tag='validation')

---
## ⑤ Combine Batches per Buffer & Upload

Run **after all extraction cells above are complete**.

In [ ]:
import glob

def combine_batches(buffer_m, tag):
    """Concatenate all batch CSVs for a given buffer size and tag."""
    pattern = f"landsat_{buffer_m}m_{tag}_batch_*.csv"
    files = sorted(glob.glob(pattern))
    if not files:
        print(f"⚠️  No files found matching: {pattern}")
        return None
    dfs = [pd.read_csv(f) for f in files]
    combined = pd.concat(dfs, ignore_index=True)
    out_name = f"landsat_features_{tag}_{buffer_m}m.csv"
    combined.to_csv(out_name, index=False)
    print(f"  ✅ {out_name}  — {combined.shape[0]} rows from {len(files)} batches")
    return combined

print("Combining training batches...")
for buf in [50, 150, 200]:
    combine_batches(buf, 'training')

print("\nCombining validation batches...")
for buf in [50, 150, 200]:
    combine_batches(buf, 'validation')

print("\n✅ All buffer files combined.")

In [ ]:
# ── EXPORT STAGING TABLES TO CSV ─────────────────────────────────────────────
# Run this cell AFTER the Mega Batcher is done (or if you want to check progress).

for buffer_m in [50, 150, 200]:
    table_name = f"LANDSAT_STAGING_TRAINING_{buffer_m}M"
    try:
        print(f"Exporting {table_name}...")
        df = session.table(table_name).to_pandas()
        out_name = f"landsat_features_training_{buffer_m}m.csv"
        df.to_csv(out_name, index=False)
        print(f"  ✅ Created: {out_name} ({len(df)} rows)")
    except:
        print(f"  ⚠️  Table {table_name} not found yet.")

In [ ]:
from IPython.display import HTML, display

# Create links for Training Data
print("🔗 Download Training Data CSVs:")
for buffer_m in [50, 150, 200]:
    fname = f"landsat_features_training_{buffer_m}m.csv"
    html = f'<a href="{fname}" target="_blank">Download {buffer_m}m Training: {fname}</a>'
    display(HTML(html))

# Create links for Validation Data
print("\n🔗 Download Validation Data CSVs:")
for buffer_m in [50, 150, 200]:
    fname = f"landsat_features_validation_{buffer_m}m.csv"
    html = f'<a href="{fname}" target="_blank">Download {buffer_m}m Validation: {fname}</a>'
    display(HTML(html))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# DOWNLOAD SOLUTION FOR CSV FILES (SNOWFLAKE COMPATIBLE)
# ═══════════════════════════════════════════════════════════════════════════════

import glob
from IPython.display import Javascript, display

print("🔍 Checking for available CSV files in Snowflake workspace...")

# Check what CSV files exist in the current directory
try:
    csv_files = glob.glob("*.csv")
    landsat_files = [f for f in csv_files if 'landsat' in f.lower()]
    terraclimate_files = [f for f in csv_files if 'terraclimate' in f.lower()]

    print(f"\n📊 Found {len(csv_files)} total CSV files:")
    print(f"   🛰️  Landsat files: {len(landsat_files)}")
    print(f"   🌍 TerraClimate files: {len(terraclimate_files)}")

    if csv_files:
        print("\n📁 Available files:")
        for file in sorted(csv_files):
            try:
                # Try to get file size, but don't fail if we can't
                size_mb = len(open(file, 'r').read()) / (1024*1024)
                print(f"   • {file} ({size_mb:.1f} MB)")
            except:
                print(f"   • {file} (size unknown)")
        
        print("\n" + "="*70)
        print("💡 SNOWFLAKE DOWNLOAD OPTIONS:")
        print("="*70)
        print("Option 1: Use Snowflake's 'Download' feature in the Files panel")
        print("Option 2: Export files through Snowflake's file management interface")
        print("Option 3: Use Snowflake's stage commands to download to local storage")
        print("Option 4: Access files through Snowflake's web interface file browser")
        print("\n🔧 For Snowflake Notebook Environment:")
        print("   • Files are stored in your Snowflake session workspace")
        print("   • Use the Snowflake UI to export/download CSV files")
        print("   • Check the 'Files' or 'Results' tab in your Snowflake interface")
        
    else:
        print("\n⚠️  No CSV files found. Please run the extraction cells first.")
        
except Exception as e:
    print(f"\n⚠️  Could not access file system: {e}")
    print("This is expected in Snowflake environment.")
    print("\n💡 SNOWFLAKE EXTRACTION STATUS:")
    print("   • Check your Snowflake staging tables:")
    print("     - LANDSAT_STAGING_TRAINING_50M")
    print("     - LANDSAT_STAGING_TRAINING_150M") 
    print("     - LANDSAT_STAGING_TRAINING_200M")
    print("   • Use Snowflake's SQL interface to export results")
    print("   • Files should be available in your Snowflake session workspace")

print(f"\n🚀 Expected Landsat files:")
print(f"   • landsat_features_training_50m.csv")
print(f"   • landsat_features_training_150m.csv")
print(f"   • landsat_features_training_200m.csv")